# 1. Imports & Setup

In [4]:

import glob
import os
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb






# Load all CSV files from Colab environment directory

In [5]:
file_paths = glob.glob("/content/*.csv")
dfs = [pd.read_csv(file) for file in file_paths]
print(f"Successfully loaded {len(dfs)} datasets into memory.")

Successfully loaded 20 datasets into memory.


# 2. Data Loading & Preprocessing

In [6]:

inv = pd.read_csv("/content/Inventory_History.csv")
sales = pd.read_csv("/content/Sales_Transactions.csv")
loc = pd.read_csv("/content/Locations.csv")
bom_df = pd.read_csv("/content/Product_BOM.csv")
product_master = pd.read_csv("/content/Product_Master.csv")


# Convert date fields to period/datetime format

In [7]:


inv["month"] = pd.to_datetime(inv["Snapshot_Month"]).dt.to_period("M")
sales["month"] = pd.to_datetime(sales["Month_Key"]).dt.to_period("M")
sales["Order_Date"] = pd.to_datetime(sales["Order_Date"])

KEYS = ["Location_ID", "Product_ID"]
inv = inv.sort_values([*KEYS, "month"]).reset_index(drop=True)
inv["dim"] = inv["month"].dt.days_in_month


# 3. Feature Engineering for Global LightGBM Model

In [8]:
def create_lags_and_features(df):
    df = df.sort_values([*KEYS, "month"]).reset_index(drop=True)
    g = df.groupby(KEYS)

    df["lag_1"] = g["Sold_Qty"].shift(1)
    df["lag_2"] = g["Sold_Qty"].shift(2)
    df["lag_12"] = g["Sold_Qty"].shift(12)

    df["rolling_mean_3"] = g["Sold_Qty"].shift(1).rolling(3).mean()
    df["rolling_std_3"] = g["Sold_Qty"].shift(1).rolling(3).std()

    df["lead_time"] = df["Supplier_Lead_Time_Days"]
    df["month_num"] = df["month"].dt.month

    return df

feature_df = create_lags_and_features(inv.copy())

# 4. Train LightGBM Quantile Regression Models

In [9]:

features = [
    "lag_1",
    "lag_2",
    "lag_12",
    "rolling_mean_3",
    "rolling_std_3",
    "lead_time",
    "month_num",
]
target = "Sold_Qty"

train_mask = feature_df["month"] < pd.Period("2025-10")
test_mask = feature_df["month"] >= pd.Period("2025-10")

X_train, y_train = (
    feature_df.loc[train_mask, features],
    feature_df.loc[train_mask, target],
)
X_test, y_test = (
    feature_df.loc[test_mask, features],
    feature_df.loc[test_mask, target],
)

for alpha in [0.5, 0.9]:
    model = lgb.LGBMRegressor(
        objective="quantile",
        alpha=alpha,
        boosting_type="gbdt",
        n_estimators=200,
        learning_rate=0.05,
        random_state=42,
        verbose=-1,
    )
    model.fit(X_train, y_train)
    feature_df.loc[test_mask, f"pred_q_{alpha}"] = model.predict(X_test)

print("LightGBM Quantile Models trained successfully.")




LightGBM Quantile Models trained successfully.


# 5. Order-Up-To Policy Simulation with Upstream Lead Times

In [10]:
def simulate_order_up_to_policy(df_test):
    simulation_results = []

    for (loc_id, prod_id), group in df_test.groupby(KEYS):
        group = group.sort_values("month")
        on_hand = group.iloc[0]["Opening_Stock"]
        pipeline_orders = []

        for idx, row in group.iterrows():
            current_month = row["month"]
            actual_demand = row["Sold_Qty"]
            lead_time_days = row["Supplier_Lead_Time_Days"]
            lead_time_months = max(1, round(lead_time_days / 30))

            arriving_qty = sum(
                [
                    qty
                    for arrival, qty in pipeline_orders
                    if arrival <= current_month
                ]
            )
            pipeline_orders = [
                (arr, q) for arr, q in pipeline_orders if arr > current_month
            ]
            on_hand += arriving_qty

            sold = min(on_hand, actual_demand)
            lost_sales = max(0, actual_demand - on_hand)
            on_hand -= sold

            q_forecast = row.get("pred_q_0.9", row["Avg_Daily_Sales_90D"] * 30)
            target_inventory = q_forecast * (lead_time_months + 1)

            current_pipeline_qty = sum([q for _, q in pipeline_orders])
            inventory_position = on_hand + current_pipeline_qty
            order_qty = max(0, target_inventory - inventory_position)

            arrival_month = current_month + lead_time_months
            if order_qty > 0:
                pipeline_orders.append((arrival_month, order_qty))

            simulation_results.append(
                {
                    "month": current_month,
                    "Location_ID": loc_id,
                    "Product_ID": prod_id,
                    "Ending_Stock": on_hand,
                    "Sold_Qty": sold,
                    "Lost_Sales": lost_sales,
                    "Order_Placed": order_qty,
                }
            )

    return pd.DataFrame(simulation_results)


sim_df = simulate_order_up_to_policy(feature_df[test_mask])
print(
    f"Inventory Simulation Complete. Total Lost Sales: {sim_df['Lost_Sales'].sum():,.2f}"
)




Inventory Simulation Complete. Total Lost Sales: 18,151.92


# 6. Upstream BOM & Raw Material Requirements Integration

In [11]:
def check_bom_constraints(sim_results_df, bom_df):
    merged = sim_results_df.merge(bom_df, on="Product_ID", how="left")
    merged["Required_Material_Qty"] = (
        merged["Order_Placed"] * merged["Required_Per_Unit"]
    )
    return (
        merged.groupby(["month", "Material_ID", "Material_Name", "Unit"])[
            "Required_Material_Key"
            if "Required_Material_Key" in merged.columns
            else "Required_Material_Qty"
        ]
        .sum()
        .reset_index()
    )

# 7. Export Data for External Web App


In [12]:
os.makedirs("app_data", exist_ok=True)
export_sim = sim_df.copy()
export_sim["month"] = export_sim["month"].astype(str)
export_sim.to_json("app_data/sim_results.json", orient="records")
product_master.to_json("app_data/product_master.json", orient="records")
print("Data exported successfully to 'app_data/' for external frontend UI.")

Data exported successfully to 'app_data/' for external frontend UI.
